In [2]:
import os
import pandas as pd

# This tells Python to look outside the 'notebooks' folder and into 'data/raw'
data_dir = "../data/raw/"

# The exact list of your 10 files
csv_files = [
    "01_fund_master (1).csv",
    "02_nav_history.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "07_scheme_performance.csv",
    "08_investor_transactions.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

# Loop through each file and print its details
for file_name in csv_files:
    file_path = os.path.join(data_dir, file_name)
    print("=" * 60)
    print(f"FILE: {file_name}")
    print("=" * 60)
    
    try:
        df = pd.read_csv(file_path)
        
        print(f"Rows: {df.shape} | Columns: {df.shape}\n")
        print("Column Types:")
        print(df.dtypes)
        print("\nFirst 3 Rows:")
        print(df.head(3))
        print("\n\n")
        
    except Exception as e:
        print(f"Error reading {file_name}. Error: {e}\n\n")

FILE: 01_fund_master (1).csv
Rows: (40, 15) | Columns: (40, 15)

Column Types:
amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object

First 3 Rows:
   amfi_code       fund_house                                 scheme_name  \
0     119551  SBI Mutual Fund   SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund    SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund  SBI Small Cap Fund - Regular Plan - Growth   

  category sub_category     plan launch_date             benchmark  \
0   Equity    Large Cap  Regular  2006-02-14         NIFTY 100 

In [3]:
import os
import pandas as pd

data_dir = "../data/raw/"

# 1. Load just the two files we need for this check
fund_master = pd.read_csv(os.path.join(data_dir, "01_fund_master (1).csv"))
nav_history = pd.read_csv(os.path.join(data_dir, "02_nav_history.csv"))

# 2. Explore the Fund Master Data (As requested in your project instructions)
print("=" * 60)
print("FUND MASTER METADATA SUMMARY")
print("=" * 60)
print(f"Total Unique Fund Houses: {fund_master['fund_house'].nunique()}")
print(f"Total Categories: {fund_master['category'].nunique()}")
print(f"Total Sub-Categories: {fund_master['sub_category'].nunique()}")

# Note: Using 'risk_category' based on standard dataset naming
if 'risk_category' in fund_master.columns:
    print(f"Risk Grades Available: {fund_master['risk_category'].dropna().unique()}")
elif 'risk_grade' in fund_master.columns:
    print(f"Risk Grades Available: {fund_master['risk_grade'].dropna().unique()}")

# 3. Perform the Referential Integrity Validation
print("\n" + "=" * 60)
print("DATA QUALITY: REFERENTIAL INTEGRITY CHECK")
print("=" * 60)

# Extract the unique AMFI codes from both datasets into Python sets for comparison
master_codes = set(fund_master['amfi_code'].unique())
nav_codes = set(nav_history['amfi_code'].unique())

# Find any codes that are in the Master file but missing from the NAV history
missing_codes = master_codes - nav_codes
missing_count = len(missing_codes)

print(f"Total AMFI Codes in Master File: {len(master_codes)}")
print(f"Total Unique AMFI Codes in NAV History: {len(nav_codes)}")
print("-" * 60)

if missing_count == 0:
    print("STATUS: PASS")
    print("Excellent! Every single AMFI code in the Master file has corresponding historical NAV data.")
else:
    print("STATUS: WARNING")
    print(f"There are {missing_count} mutual funds in the Master file with NO historical data.")

FUND MASTER METADATA SUMMARY
Total Unique Fund Houses: 10
Total Categories: 2
Total Sub-Categories: 12
Risk Grades Available: ['Moderate' 'Very High' 'Low' 'High' 'Moderately High']

DATA QUALITY: REFERENTIAL INTEGRITY CHECK
Total AMFI Codes in Master File: 40
Total Unique AMFI Codes in NAV History: 40
------------------------------------------------------------
STATUS: PASS
Excellent! Every single AMFI code in the Master file has corresponding historical NAV data.


In [4]:
import os
import requests
import pandas as pd
import time

data_dir = "../data/raw/"

# Dictionary containing the 6 mutual funds requested in your assignment
target_schemes = {
    "125497": "HDFC Top 100 Direct",
    "119551": "SBI Bluechip",
    "120503": "ICICI Bluechip",
    "118632": "Nippon Large Cap",
    "119092": "Axis Bluechip",
    "120841": "Kotak Bluechip"
}

print("=" * 60)
print("STARTING LIVE NAV DATA EXTRACTION VIA API")
print("=" * 60)

for amfi_code, fund_name in target_schemes.items():
    print(f"Fetching data for: {fund_name} (Code: {amfi_code})...")
    
    # 1. Define the specific API endpoint for the fund
    url = f"https://api.mfapi.in/mf/{amfi_code}"
    
    try:
        # 2. Make the HTTP GET request
        response = requests.get(url)
        
        # 3. Check if the request was successful (Status Code 200)
        if response.status_code == 200:
            json_data = response.json()
            
            # 4. Extract the 'data' array containing the historical NAV points
            if "data" in json_data and len(json_data["data"]) > 0:
                nav_data = json_data["data"]
                
                # 5. Convert the nested JSON list into a Pandas DataFrame
                df = pd.DataFrame(nav_data)
                
                # Add the AMFI code as a column to ensure consistency with our local datasets
                df['amfi_code'] = amfi_code
                df = df[['amfi_code', 'date', 'nav']]
                
                # 6. Save the DataFrame as a CSV in the raw data folder
                file_name = f"live_nav_{amfi_code}.csv"
                file_path = os.path.join(data_dir, file_name)
                df.to_csv(file_path, index=False)
                
                print(f"Success: Saved {len(df)} records to {file_name}")
            else:
                print(f"Warning: No data found in the API response for {amfi_code}.")
        else:
            print(f"Failed to fetch {amfi_code}. HTTP Status code: {response.status_code}")
            
    except Exception as e:
        print(f"An error occurred while processing {amfi_code}: {e}")
        
    # Introduce a one-second pause between requests to prevent server blocking
    time.sleep(1)

print("-" * 60)
print("API Extraction Complete.")

STARTING LIVE NAV DATA EXTRACTION VIA API
Fetching data for: HDFC Top 100 Direct (Code: 125497)...
Success: Saved 3092 records to live_nav_125497.csv
Fetching data for: SBI Bluechip (Code: 119551)...
Success: Saved 3237 records to live_nav_119551.csv
Fetching data for: ICICI Bluechip (Code: 120503)...
Success: Saved 3308 records to live_nav_120503.csv
Fetching data for: Nippon Large Cap (Code: 118632)...
Success: Saved 3299 records to live_nav_118632.csv
Fetching data for: Axis Bluechip (Code: 119092)...
Success: Saved 3566 records to live_nav_119092.csv
Fetching data for: Kotak Bluechip (Code: 120841)...
Success: Saved 3302 records to live_nav_120841.csv
------------------------------------------------------------
API Extraction Complete.
